### [ex01] Functions as values, closures and decorators

In [ ]:
import math
import time
from collections.abc import Callable

# a function is an object: it can be passed, stored and returned like any other value
print(type(math.sqrt), math.sqrt.__name__)


def apply_twice(f: Callable[[float], float], x: float) -> float:
    return f(f(x))


print(apply_twice(math.sqrt, 16.0))
print(apply_twice(lambda x: 2 * x + 1, 1.0))

fs = [math.sin, math.cos, math.exp, lambda x: x ** 2]
print([f(0.0) for f in fs])
print([f.__name__ for f in fs])

In [ ]:
def make_power(n: int) -> Callable[[float], float]:
    """Return the function x -> x ** n."""

    def power(x: float) -> float:
        return x ** n  # n is a free variable: it lives in the enclosing scope

    return power


square = make_power(2)
cube = make_power(3)
print(square(3.0), cube(3.0))
print(square.__closure__[0].cell_contents)  # the value captured by the closure

In [ ]:
def make_counter() -> Callable[[], int]:
    count = 0

    def counter() -> int:
        nonlocal count  # without it, `count += 1` would create a new local variable
        count += 1
        return count

    return counter


c1 = make_counter()
c2 = make_counter()
print(c1(), c1(), c1(), c2())  # each closure has its own count

In [ ]:
def timed(f: Callable) -> Callable:
    """A decorator: takes a function, returns a function that measures every call."""

    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = f(*args, **kwargs)
        print(f"{f.__name__}: {time.perf_counter() - start:.4f} s")
        return result

    return wrapper


def sum_to(n: int) -> int:
    return sum(range(n))


sum_to = timed(sum_to)  # the name now refers to the wrapper
print(sum_to(10_000_000))


@timed  # the same as: sum_squares = timed(sum_squares)
def sum_squares(n: int) -> int:
    """Sum of i * i for i < n."""
    return sum(i * i for i in range(n))


print(sum_squares(1_000_000))
print(sum_squares.__name__, sum_squares.__doc__)  # the wrapper hides the original

In [ ]:
from functools import wraps


def repeat(times: int) -> Callable:
    """A decorator with a parameter: repeat(3) returns the actual decorator."""

    def decorator(f: Callable) -> Callable:
        @wraps(f)  # copies __name__, __doc__, ... from f to wrapper
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = f(*args, **kwargs)
            return result

        return wrapper

    return decorator


@repeat(3)
def greet(name: str) -> str:
    """Say hello."""
    print(f"hello, {name}")
    return name


print(greet("Ada"))
print(greet.__name__, greet.__doc__)

In [ ]:
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(timed(fib)(30))


def cached(f: Callable) -> Callable:
    """A decorator that remembers the result of every call (memoization)."""
    raise NotImplementedError  # TODO


# the standard library has it: functools.cache (and lru_cache with a size limit)
from functools import cache


@cache
def fib_cached(n: int) -> int:
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)


print(timed(fib_cached)(30))
print(fib_cached.cache_info())

### [ex02] Derivatives by finite differences: truncation error and rounding error

In [ ]:
import math
from collections.abc import Callable

Function = Callable[[float], float]


def forward_difference(f: Function, x: float, h: float) -> float:
    return (f(x + h) - f(x)) / h


def central_difference(f: Function, x: float, h: float) -> float:
    return (f(x + h) - f(x - h)) / (2 * h)


x0 = 1.0
exact = math.cos(x0)  # (sin x)' = cos x

print(f"{'h':>7}  {'forward error':>14}  {'central error':>14}")
for k in range(1, 17):
    h = 10.0 ** -k
    e_fwd = abs(forward_difference(math.sin, x0, h) - exact)
    e_ctr = abs(central_difference(math.sin, x0, h) - exact)
    print(f"{h:7.0e}  {e_fwd:14.2e}  {e_ctr:14.2e}")

In [ ]:
import matplotlib.pyplot as plt

hs = [10.0 ** -k for k in range(1, 17)]
errors_fwd = [abs(forward_difference(math.sin, x0, h) - exact) for h in hs]
errors_ctr = [abs(central_difference(math.sin, x0, h) - exact) for h in hs]

plt.loglog(hs, errors_fwd, "o-", label="forward")
plt.loglog(hs, errors_ctr, "s-", label="central")
plt.loglog(hs, hs, ":", color="grey", label="h")
plt.loglog(hs, [h * h for h in hs], "--", color="grey", label="h²")
plt.loglog(hs, [2.2e-16 / h for h in hs], "-.", color="grey", label="ε / h")
plt.xlabel("h")
plt.ylabel("absolute error of f'(1), f = sin")
plt.legend()
plt.show()

In [ ]:
def observed_order(method, f: Function, x: float, exact: float, h1: float, h2: float) -> float:
    """The observed order p: error(h) ~ C * h**p, so p = log(e1 / e2) / log(h1 / h2)."""
    e1 = abs(method(f, x, h1) - exact)
    e2 = abs(method(f, x, h2) - exact)
    return math.log(e1 / e2) / math.log(h1 / h2)


print(observed_order(forward_difference, math.sin, x0, exact, 1e-2, 1e-3))
print(observed_order(central_difference, math.sin, x0, exact, 1e-2, 1e-3))

In [ ]:
def derivative(f: Function, h: float = 1e-5) -> Function:
    """Return the function x -> f'(x), approximated by the central difference."""
    raise NotImplementedError  # TODO


# cos_approx = derivative(math.sin)
# print(cos_approx(0.0), cos_approx(math.pi))


def second_difference(f: Function, x: float, h: float) -> float:
    """f''(x) ~ (f(x + h) - 2 f(x) + f(x - h)) / h**2"""
    raise NotImplementedError  # TODO

In [ ]:
def taylor_exp(x: float, n: int) -> float:
    """The Taylor polynomial of exp at 0: the sum of x**k / k! for k = 0..n."""
    total = 0.0
    term = 1.0  # x**0 / 0!
    for k in range(1, n + 1):
        total += term
        term *= x / k  # x**k / k! from x**(k - 1) / (k - 1)!
    return total + term


for n in [1, 2, 4, 8, 16]:
    print(n, taylor_exp(1.0, n), abs(taylor_exp(1.0, n) - math.e))

In [ ]:
# truncation error: how many terms does x = 10 need?
for n in [10, 20, 30, 40]:
    print(n, taylor_exp(10.0, n), math.exp(10.0))

# rounding error: the same series at x = -20
print(taylor_exp(-20.0, 100), math.exp(-20.0))
print(1 / taylor_exp(20.0, 100), math.exp(-20.0))

### [ex03] Classes, data classes and special methods: a vector in the plane

In [ ]:
import math


class Vector2D:
    """A vector in the plane, written by hand."""

    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Vector2D({self.x}, {self.y})"

    def __eq__(self, other) -> bool:
        return isinstance(other, Vector2D) and (self.x, self.y) == (other.x, other.y)

    def __add__(self, other: "Vector2D") -> "Vector2D":
        return Vector2D(self.x + other.x, self.y + other.y)

    def __mul__(self, k: float) -> "Vector2D":  # v * k
        return Vector2D(k * self.x, k * self.y)

    def __rmul__(self, k: float) -> "Vector2D":  # k * v
        return self * k

    def __abs__(self) -> float:
        return math.hypot(self.x, self.y)


u = Vector2D(3.0, 4.0)
v = Vector2D(1.0, 2.0)
print(u, u + v, 2 * u, u * 2, abs(u))
print(u == Vector2D(3.0, 4.0), u is Vector2D(3.0, 4.0))
print(u * v)  # no error, but what is it?

In [ ]:
class Point:
    """The same data without the special methods."""

    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y


p = Point(1.0, 2.0)
print(p)
print(p == Point(1.0, 2.0))  # the default __eq__ compares identity
try:
    p + p
except TypeError as err:
    print("TypeError:", err)

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Vec:
    """The vector as a data class: __init__, __repr__, __eq__ and __hash__ are generated."""

    x: float
    y: float

    def __add__(self, other: "Vec") -> "Vec":
        return Vec(self.x + other.x, self.y + other.y)

    def __mul__(self, k: float) -> "Vec":
        if not isinstance(k, (int, float)):
            return NotImplemented  # Python then tries k.__rmul__, and raises TypeError if that fails
        return Vec(self.x * k, self.y * k)

    __rmul__ = __mul__

    def __abs__(self) -> float:
        return math.hypot(self.x, self.y)

    def __matmul__(self, other: "Vec") -> float:  # u @ v: the dot product
        return self.x * other.x + self.y * other.y

    def __sub__(self, other: "Vec") -> "Vec":
        raise NotImplementedError  # TODO

    def __neg__(self) -> "Vec":
        raise NotImplementedError  # TODO

    def normalized(self) -> "Vec":
        """The unit vector with the same direction."""
        raise NotImplementedError  # TODO


a = Vec(3.0, 4.0)
b = Vec(1.0, 2.0)
print(a, a + b, 2 * a, abs(a), a @ b)
print(a == Vec(3.0, 4.0))
try:
    a * b
except TypeError as err:
    print("TypeError:", err)
try:
    a.x = 10.0
except AttributeError as err:  # FrozenInstanceError is a subclass of AttributeError
    print("frozen:", err)
print({a, Vec(3.0, 4.0), b})  # hashable, so usable in a set or as a dict key

In [ ]:
@dataclass(frozen=True, order=True)
class OrderedVec:
    x: float
    y: float


vs = [OrderedVec(3.0, 4.0), OrderedVec(1.0, 2.0), OrderedVec(1.0, 10.0)]
print(sorted(vs))  # order=True compares (x, y) as tuples
print(sorted(vs, key=lambda v: math.hypot(v.x, v.y)))  # order by length
print(OrderedVec(1.0, 10.0) < OrderedVec(3.0, 4.0))

In [ ]:
from collections import namedtuple

NTVec = namedtuple("NTVec", ["x", "y"])
n = NTVec(3.0, 4.0)
print(n, n.x, n[0], tuple(n))

In [ ]:
def cosine_similarity(u: Vec, v: Vec) -> float:
    """The cosine of the angle between u and v."""
    raise NotImplementedError  # TODO


# print(cosine_similarity(Vec(1.0, 0.0), Vec(1.0, 1.0)))  # 0.7071...

### [ex04] Dual numbers: exact derivatives by forward-mode automatic differentiation

In [ ]:
import math
from dataclasses import dataclass


@dataclass(frozen=True)
class Dual:
    """a + b*eps with eps**2 = 0: `real` is the value, `dual` carries the derivative."""

    real: float
    dual: float = 0.0

    def __add__(self, other) -> "Dual":
        other = lift(other)
        return Dual(self.real + other.real, self.dual + other.dual)

    __radd__ = __add__

    def __mul__(self, other) -> "Dual":
        other = lift(other)
        # (a + b eps)(c + d eps) = ac + (ad + bc) eps, because eps**2 = 0: the product rule
        return Dual(self.real * other.real, self.real * other.dual + self.dual * other.real)

    __rmul__ = __mul__

    def __neg__(self) -> "Dual":
        return Dual(-self.real, -self.dual)

    def __sub__(self, other) -> "Dual":
        return self + (-lift(other))

    def __rsub__(self, other) -> "Dual":
        return lift(other) - self

    def __truediv__(self, other) -> "Dual":
        """(a + b eps) / (c + d eps): the quotient rule."""
        raise NotImplementedError  # TODO

    def __rtruediv__(self, other) -> "Dual":
        return lift(other) / self

    def __pow__(self, n: int) -> "Dual":
        """(a + b eps) ** n for an integer n: the power rule."""
        raise NotImplementedError  # TODO


def lift(x) -> Dual:
    """A plain number is a dual number with no derivative part."""
    return x if isinstance(x, Dual) else Dual(float(x))


x = Dual(3.0, 1.0)  # dual part 1: x changes at rate 1 with respect to itself
print(x + 1, 2 * x, x * x)
print((x + 2) * (x + 1))  # f(x) = (x + 2)(x + 1) = x**2 + 3x + 2, f'(3) = 2 * 3 + 3 = 9

In [ ]:
def derivative(f):
    """Return the function x -> f'(x): evaluate f at x + 1 eps and read the dual part."""
    return lambda x: f(Dual(x, 1.0)).dual


def poly(x):
    return x * x * x - 2 * x + 1  # f'(x) = 3 x**2 - 2


print(derivative(poly)(2.0), 3 * 2.0 ** 2 - 2)

In [ ]:
# the chain rule: f(a + b eps) = f(a) + f'(a) b eps
def sin(x):
    if not isinstance(x, Dual):
        return math.sin(x)
    return Dual(math.sin(x.real), math.cos(x.real) * x.dual)


def cos(x):
    if not isinstance(x, Dual):
        return math.cos(x)
    return Dual(math.cos(x.real), -math.sin(x.real) * x.dual)


def exp(x):
    if not isinstance(x, Dual):
        return math.exp(x)
    return Dual(math.exp(x.real), math.exp(x.real) * x.dual)


def log(x):
    raise NotImplementedError  # TODO


def g(x):
    return sin(x) * exp(-x)  # g'(x) = (cos x - sin x) exp(-x)


x0 = 1.5
print(derivative(g)(x0), (math.cos(x0) - math.sin(x0)) * math.exp(-x0))

In [ ]:
# three ways to a derivative: the formula, a finite difference, a dual number
def central_difference(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)


exact = (math.cos(x0) - math.sin(x0)) * math.exp(-x0)
print(f"formula            {exact:.17f}")
print(f"central, h = 1e-5  {central_difference(g, x0, 1e-5):.17f}")
print(f"dual number        {derivative(g)(x0):.17f}")
print(abs(central_difference(g, x0, 1e-5) - exact), abs(derivative(g)(x0) - exact))

In [ ]:
def newton(f, x0: float, steps: int = 6) -> float:
    """Newton's method x <- x - f(x) / f'(x), with f' from dual numbers."""
    df = derivative(f)
    x = x0
    for _ in range(steps):
        x = x - f(x) / df(x)
        print(x)
    return x


print(newton(lambda x: x * x - 2, 1.0), math.sqrt(2))

### [ex05] Exceptions and context managers

In [ ]:
import math


def safe_log(x: float) -> float:
    try:
        return math.log(x)
    except ValueError as err:  # math domain error
        print(f"ValueError: {err}")
        return math.nan


print(safe_log(math.e), safe_log(-1.0), safe_log(0.0))

In [ ]:
def parse_number(text: str) -> float:
    try:
        value = float(text)
    except ValueError:
        print(f"not a number: {text!r}")
        raise  # the same exception continues to the caller
    else:
        print("parsed", value)  # only when nothing was raised
        return value
    finally:
        print(f"done with {text!r}")  # always: after return and after raise


parse_number("3.14")
try:
    parse_number("abc")
except ValueError:
    print("caught again, outside")

In [ ]:
# the hierarchy: except catches the class and its subclasses
print(issubclass(ZeroDivisionError, ArithmeticError), issubclass(OverflowError, ArithmeticError))
for expr in ["1 / 0", "math.exp(1000)", "int('x')", "[1, 2][5]", "{}['k']"]:
    try:
        eval(expr)
    except ArithmeticError as err:
        print(f"{expr:16} -> ArithmeticError: {type(err).__name__}")
    except (LookupError, ValueError) as err:
        print(f"{expr:16} -> {type(err).__name__}: {err}")

In [ ]:
class DomainError(ValueError):
    """Raised when a function is evaluated outside its domain."""


def sqrt_checked(x: float) -> float:
    if x < 0:
        raise DomainError(f"square root of a negative number: {x}")
    return math.sqrt(x)


try:
    sqrt_checked(-4.0)
except DomainError as err:
    print(type(err).__name__, err)


def mean(xs: list[float]) -> float:
    try:
        return sum(xs) / len(xs)
    except ZeroDivisionError as err:
        raise ValueError("mean of an empty list") from err  # the cause stays attached


try:
    mean([])
except ValueError as err:
    print(err, "| caused by:", repr(err.__cause__))

In [ ]:
# a file is a resource: `with` closes it, also when the block raises
hs = [10.0 ** -k for k in range(1, 9)]
with open("errors.csv", "w", encoding="utf-8") as out:
    out.write("h,error\n")
    for h in hs:
        err = abs((math.sin(1 + h) - math.sin(1 - h)) / (2 * h) - math.cos(1))
        out.write(f"{h:.0e},{err:.3e}\n")
print(out.closed)

with open("errors.csv", encoding="utf-8") as src:
    for line in src:
        print(line.rstrip())

In [ ]:
import time


class Timer:
    """with Timer() as t: ...   then t.elapsed holds the seconds."""

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, traceback):
        self.elapsed = time.perf_counter() - self.start
        status = f" (after {exc_type.__name__})" if exc_type else ""
        print(f"elapsed: {self.elapsed:.4f} s{status}")
        return False  # False: the exception, if any, continues


with Timer() as t:
    total = sum(i * i for i in range(1_000_000))
print(t.elapsed)

try:
    with Timer():
        1 / 0
except ZeroDivisionError:
    print("the exception passed through __exit__")

In [ ]:
import os
from contextlib import contextmanager


@contextmanager
def timer(label: str):
    start = time.perf_counter()
    try:
        yield  # paused while the with block runs in the caller
    finally:
        print(f"{label}: {time.perf_counter() - start:.4f} s")


with timer("sum of squares"):
    total = sum(i * i for i in range(1_000_000))


@contextmanager
def working_directory(path: str):
    """Run the block inside `path`, then come back, also after an exception."""
    raise NotImplementedError  # TODO


# with working_directory(".."):
#     print(os.getcwd())
# print(os.getcwd())

### [ex06] Iterators, generators and type annotations

In [ ]:
import itertools
import math
import sys
from collections.abc import Iterable, Iterator


def taylor_terms(x: float) -> Iterator[float]:
    """x**k / k! for k = 0, 1, 2, ...: an infinite sequence, one term at a time."""
    term = 1.0
    k = 0
    while True:
        yield term  # pause here; resume at the next request
        k += 1
        term *= x / k


terms = taylor_terms(1.0)
print(type(terms))
print(next(terms), next(terms), next(terms))
print(list(itertools.islice(taylor_terms(1.0), 6)))

In [ ]:
# the iterator protocol: iter() returns an iterator, next() returns the elements,
# StopIteration ends the sequence; a for loop does exactly this
it = iter([10, 20, 30])
print(type(it), next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration: a for loop stops here")
print(iter(terms) is terms)  # a generator is its own iterator


class Countdown:
    """What a generator does by hand: __iter__ returns the iterator, __next__ the elements."""

    def __init__(self, start: int):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        self.current -= 1
        return self.current + 1


print(list(Countdown(3)))
for i in Countdown(2):  # for calls iter() once, then next() until StopIteration
    print(i)

In [ ]:
# partial sums: the Taylor polynomials of exp, one after another
sums = itertools.accumulate(taylor_terms(1.0))
for n, s in enumerate(itertools.islice(sums, 12)):
    print(n, s, abs(s - math.e))

In [ ]:
def exp_series(x: float, eps: float = 1e-16) -> float:
    """Sum the terms while they are larger than eps."""
    significant = itertools.takewhile(lambda t: abs(t) > eps, taylor_terms(x))
    return sum(significant)


print(exp_series(1.0), math.exp(1.0))
print(exp_series(-20.0), math.exp(-20.0))  # the rounding problem of ex02 stays

In [ ]:
squares_list = [i * i for i in range(1_000_000)]
squares_gen = (i * i for i in range(1_000_000))
print(sys.getsizeof(squares_list), sys.getsizeof(squares_gen))
print(sum(squares_gen))
print(sum(squares_gen))  # a generator is used up after one pass

In [ ]:
def batches(xs: list, size: int) -> Iterator[list]:
    """Consecutive slices of xs of length size, the last one possibly shorter (mini-batches)."""
    raise NotImplementedError  # TODO


# for b in batches(list(range(10)), 4):
#     print(b)  # [0, 1, 2, 3], [4, 5, 6, 7], [8, 9]


def running_mean(xs: Iterable[float]) -> Iterator[float]:
    """The mean of the elements seen so far, one value per element."""
    raise NotImplementedError  # TODO


# print(list(running_mean([2.0, 4.0, 6.0])))  # [2.0, 3.0, 4.0]

In [ ]:
# type annotations: documentation checked by tools (mypy, pyright), not by the interpreter
from collections.abc import Callable

Function = Callable[[float], float]


def compose(f: Function, g: Function) -> Function:
    return lambda x: f(g(x))


h = compose(math.sqrt, abs)
print(h(-16.0))
print(compose("not a function", abs))  # no error: annotations are not checked at run time
print(compose.__annotations__)

### [ex07] Tests with pytest

In [ ]:
import math

import pytest

try:
    from ex04 import Dual, derivative, exp, sin  # as a file: `pytest ex07.py` in src/lab02
except ImportError:
    pass  # in the notebook the names are already defined by ex04


def test_add_and_mul():
    x = Dual(3.0, 1.0)
    assert x + 1 == Dual(4.0, 1.0)
    assert 2 * x == Dual(6.0, 2.0)
    assert x * x == Dual(9.0, 6.0)


def test_derivative_of_polynomial():
    assert derivative(lambda x: x * x * x - 2 * x + 1)(2.0) == 10.0


@pytest.mark.parametrize("x", [0.0, 0.5, 1.0, 2.0, -3.0])
def test_derivative_of_sin_is_cos(x):
    assert derivative(sin)(x) == pytest.approx(math.cos(x))


def test_product_and_chain_rule():
    x0 = 1.5
    expected = (math.cos(x0) - math.sin(x0)) * math.exp(-x0)
    assert derivative(lambda x: sin(x) * exp(-x))(x0) == pytest.approx(expected)


def test_division():
    assert derivative(lambda x: 1 / x)(2.0) == pytest.approx(-0.25)


def test_division_by_zero():
    with pytest.raises(ZeroDivisionError):
        Dual(1.0, 1.0) / Dual(0.0, 1.0)

In [ ]:
def run_tests(namespace: dict) -> None:
    """A small stand-in for pytest inside a notebook: calls every test_* function."""
    for name, f in list(namespace.items()):
        if not name.startswith("test_") or not callable(f):
            continue
        marks = [m for m in getattr(f, "pytestmark", []) if m.name == "parametrize"]
        argsets = [v if isinstance(v, tuple) else (v,) for v in marks[0].args[1]] if marks else [()]
        for args in argsets:
            try:
                f(*args)
                print(f"PASS  {name}{args if args else ''}")
            except Exception as err:
                print(f"FAIL  {name}{args if args else ''}: {type(err).__name__}: {err}")


if __name__ == "__main__":  # true in the notebook and for `python ex07.py`, not under pytest
    run_tests(globals())